In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/vibecoding-agents-capstone-project/NOTE.md


In [55]:
##reconFlow AI: Multi-Agent Invoice Reconciliation System with Tool Calling and RAG-Based Decision Intelligence

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 43.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 71.5 MB/s eta 0:00:00:00:010:01


In [4]:
# Install required libraries

!pip -q install langgraph
!pip -q install langchain
!pip -q install langchain-core
!pip -q install pydantic
!pip -q install chromadb
!pip -q install sentence-transformers
!pip -q install faker
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 712.7 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 46.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages tha

In [5]:
import json
import random
import uuid
from datetime import datetime
import pandas as pd
from faker import Faker

fake = Faker()

In [6]:
#Create Sample Purchase Orders
#We'll create realistic finance data so the agents have something to work with.

purchase_orders = pd.DataFrame({
    "PO_ID":[
        "PO1001",
        "PO1002",
        "PO1003",
        "PO1004",
        "PO1005"
    ],

    "Vendor":[
        "ABC Electronics",
        "Global Tech",
        "Office World",
        "Cloud Solutions",
        "Dell India"
    ],

    "Amount":[
        15000,
        28000,
        9500,
        45000,
        67000
    ],

    "Status":[
        "Approved",
        "Approved",
        "Pending",
        "Approved",
        "Approved"
    ]
})

purchase_orders

,PO_ID,Vendor,Amount,Status
0,PO1001,ABC Electronics,15000,Approved
1,PO1002,Global Tech,28000,Approved
2,PO1003,Office World,9500,Pending
3,PO1004,Cloud Solutions,45000,Approved
4,PO1005,Dell India,67000,Approved


In [7]:
#CREATE SAMPLE INVOICES

invoices = pd.DataFrame({

    "Invoice_ID":[
        "INV001",
        "INV002",
        "INV003",
        "INV004"
    ],

    "PO_ID":[
        "PO1001",
        "PO1002",
        "PO1003",
        "PO1005"
    ],

    "Vendor":[
        "ABC Electronics",
        "Global Tech",
        "Office World",
        "Dell India"
    ],

    "Invoice_Amount":[
        15000,
        30000,
        9500,
        70000
    ]
})

invoices

,Invoice_ID,PO_ID,Vendor,Invoice_Amount
0,INV001,PO1001,ABC Electronics,15000
1,INV002,PO1002,Global Tech,30000
2,INV003,PO1003,Office World,9500
3,INV004,PO1005,Dell India,70000


In [8]:
# INVOICE AGENT CREATION CODE

class InvoiceAgent:
    def __init__(self, invoices_df):
        self.invoices_df = invoices_df
    def extract_invoice(self,invoice_id):
        """Extract and norlamize invoice data
        """
        row = self.invoices_df[self.invoices_df["Invoice_ID"] == invoice_id]
        if row.empty:
            return {
                "status": "error",
                "message": f"Invoice {invoice_id} not found"
            }
        row=row.iloc[0]
        structure_invoice={
            "invoice_id":row["Invoice_ID"],
            "po_id":row["PO_ID"],
            "vendor":row["Vendor"],
            "amount":float(row["Invoice_Amount"]),
            "currency":"INR",
            "extracted_at":str(datetime.now())
        }
        return structure_invoice

    def extract_all(self):
        """Extract all invoices in structured format 
        """
        return[
            self.extract_invoice(inv_id)
            for inv_id in self.invoices_df["Invoice_ID"]
        ]


In [9]:
#INVOICE AGENT TESTING

invoice_agent = InvoiceAgent(invoices)
result = invoice_agent.extract_invoice("INV002")
result

{'invoice_id': 'INV002',
 'po_id': 'PO1002',
 'vendor': 'Global Tech',
 'amount': 30000.0,
 'currency': 'INR',
 'extracted_at': '2026-06-21 04:27:58.391291'}

In [10]:
# PURCHASE ORDER MATCHING AGENT
class POAgent:
    def __init__(self, po_df):
        self.po_df = po_df

    def find_po(self, po_id):
        """Fetch PO Details"""

        row = self.po_df[self.po_df["PO_ID"] == po_id]

        if row.empty:
            return None

        return row.iloc[0].to_dict()

    def match_invoice_with_po(self, invoice):
        """Compare invoice with PO"""

        po = self.find_po(invoice["po_id"])

        if po is None:
            return {
                "status": "NO_MATCH",
                "reason": "PO not found",
                "invoice": invoice,
                "po": None
            }

        issues = []

        # vendor check
        if invoice["vendor"] != po["Vendor"]:
            issues.append("Vendor mismatch")

        # amount check
        if abs(invoice["amount"] - po["Amount"]) > 0.01:
            issues.append("Amount mismatch")

        # status check
        if po["Status"] != "Approved":
            issues.append("PO not approved")

        # FINAL DECISION
        if len(issues) == 0:
            status = "MATCH"
        elif "PO not approved" in issues:
            status = "REVIEW"
        else:
            status = "PARTIAL_MATCH"

        return {
            "status": status,
            "issues": issues,
            "invoice": invoice,
            "po": po
        }

    def process_all(self, invoice_list):
        """Process multiple invoices"""
        return [
            self.match_invoice_with_po(inv)
            for inv in invoice_list
        ]

In [11]:
#testing the purchase ordering matching agent

po_agent = POAgent(purchase_orders)

invoice_agent = InvoiceAgent(invoices)

structured_invoice = invoice_agent.extract_invoice("INV002")

result = po_agent.match_invoice_with_po(structured_invoice)

result

{'status': 'PARTIAL_MATCH',
 'issues': ['Amount mismatch'],
 'invoice': {'invoice_id': 'INV002',
  'po_id': 'PO1002',
  'vendor': 'Global Tech',
  'amount': 30000.0,
  'currency': 'INR',
  'extracted_at': '2026-06-21 04:28:17.685294'},
 'po': {'PO_ID': 'PO1002',
  'Vendor': 'Global Tech',
  'Amount': 28000,
  'Status': 'Approved'}}

In [12]:
# =========================
# STEP 7: DECISION AGENT
# =========================

class DecisionAgent:

    def decide(self, match_result):
        """
        Takes PO matching result and returns final decision
        """

        status = match_result["status"]
        issues = match_result.get("issues", [])

        invoice = match_result["invoice"]
        po = match_result["po"]

        reasoning = []

        # Case 1: PO not found
        if status == "NO_MATCH":
            return {
                "decision": "REJECT",
                "confidence": 0.95,
                "reasoning": "Purchase Order not found in system",
                "invoice_id": invoice["invoice_id"]
            }

        # Case 2: No issues
        if status == "MATCH":
            return {
                "decision": "APPROVE",
                "confidence": 0.99,
                "reasoning": "Invoice matches Purchase Order exactly",
                "invoice_id": invoice["invoice_id"]
            }

        # Case 3: PO not approved
        if "PO not approved" in issues:
            reasoning.append("PO is not approved yet")
            return {
                "decision": "ESCALATE",
                "confidence": 0.85,
                "reasoning": "; ".join(reasoning),
                "invoice_id": invoice["invoice_id"]
            }

        # Case 4: Vendor mismatch
        if "Vendor mismatch" in issues:
            reasoning.append("Vendor name does not match PO record")

        # Case 5: Amount mismatch
        if "Amount mismatch" in issues:
            reasoning.append("Invoice amount differs from PO amount")

        # Final fallback logic
        if len(issues) == 1:
            decision = "ESCALATE"
            confidence = 0.80
        else:
            decision = "REJECT"
            confidence = 0.70

        return {
            "decision": decision,
            "confidence": confidence,
            "reasoning": "; ".join(reasoning),
            "invoice_id": invoice["invoice_id"],
            "issues": issues
        }

In [13]:
# TESTing the decision agent
decision_agent = DecisionAgent()

# Take one invoice
invoice_agent = InvoiceAgent(invoices)
po_agent = POAgent(purchase_orders)

structured_invoice = invoice_agent.extract_invoice("INV002")

match_result = po_agent.match_invoice_with_po(structured_invoice)

final_decision = decision_agent.decide(match_result)

final_decision

{'decision': 'ESCALATE',
 'confidence': 0.8,
 'reasoning': 'Invoice amount differs from PO amount',
 'invoice_id': 'INV002',
 'issues': ['Amount mismatch']}

In [14]:
# =========================
# STEP 8: EMAIL AGENT
# =========================

class EmailAgent:

    def generate_email(self, decision_result):
        """
        Generate a professional vendor email
        """

        decision = decision_result["decision"]
        invoice_id = decision_result["invoice_id"]
        reasoning = decision_result["reasoning"]

        if decision == "APPROVE":

            subject = f"Invoice {invoice_id} Approved"

            body = f"""
Dear Vendor,

We are pleased to inform you that Invoice {invoice_id} has been successfully verified and approved for further payment processing.

No discrepancies were found during reconciliation.

Thank you for your cooperation.

Regards,
Finance Operations Team
"""

        elif decision == "ESCALATE":

            subject = f"Invoice {invoice_id} Requires Review"

            body = f"""
Dear Vendor,

During our reconciliation process, we identified an issue with Invoice {invoice_id}.

Reason:
{reasoning}

Please review the invoice and provide the requested clarification so we can continue processing.

Regards,
Finance Operations Team
"""

        else:

            subject = f"Invoice {invoice_id} Rejected"

            body = f"""
Dear Vendor,

Invoice {invoice_id} could not be approved.

Reason:
{reasoning}

Please correct the discrepancies and submit a revised invoice.

Regards,
Finance Operations Team
"""

        return {
            "subject": subject,
            "body": body
        }

In [15]:
# EMAIL AGENT  testing
email_agent = EmailAgent()

email = email_agent.generate_email(final_decision)

print(email["subject"])
print()
print(email["body"])

Invoice INV002 Requires Review


Dear Vendor,

During our reconciliation process, we identified an issue with Invoice INV002.

Reason:
Invoice amount differs from PO amount

Please review the invoice and provide the requested clarification so we can continue processing.

Regards,
Finance Operations Team



In [16]:
# ==========================================
# STEP 9 : RECONFLOW SUPERVISOR AGENT
# ==========================================

class ReconFlowSupervisor:

    def __init__(self, invoice_agent, po_agent,
                 decision_agent, email_agent):

        self.invoice_agent = invoice_agent
        self.po_agent = po_agent
        self.decision_agent = decision_agent
        self.email_agent = email_agent

    def process_invoice(self, invoice_id):

        logger = AgentLogger()

        logger.log(
            "Supervisor",
            f"Started processing {invoice_id}"
        )

        # Step 1 - Extract Invoice
        invoice = self.invoice_agent.extract_invoice(invoice_id)

        if invoice.get("status") == "error":
            logger.log("Invoice Agent", "Invoice not found")
            logger.show()
            return invoice

        logger.log(
            "Invoice Agent",
            f"Invoice extracted for vendor '{invoice['vendor']}'"
        )

        # Step 2 - Match Purchase Order
        match = self.po_agent.match_invoice_with_po(invoice)

        logger.log(
            "PO Agent",
            f"Matching Status: {match['status']}"
        )

        if match.get("issues"):
            logger.log(
                "PO Agent",
                ", ".join(match["issues"])
            )
        else:
            logger.log(
                "PO Agent",
                "No discrepancies found"
            )

        # Step 3 - Decision
        decision = self.decision_agent.decide(match)

        logger.log(
            "Decision Agent",
            f"Decision: {decision['decision']}"
        )

        logger.log(
            "Decision Agent",
            f"Confidence: {decision['confidence']}"
        )

        # Step 4 - Email Generation
        email = self.email_agent.generate_email(decision)

        logger.log(
            "Email Agent",
            "Vendor email drafted successfully"
        )

        logger.log(
            "Supervisor",
            "Workflow completed successfully"
        )

        logger.show()

        return {
            "invoice": invoice,
            "match_result": match,
            "decision": decision,
            "email": email,
            "logs": logger.logs
        }

In [ ]:
#Testing supervisor agent
invoice_agent = InvoiceAgent(invoices)
po_agent = POAgent(purchase_orders)
decision_agent = DecisionAgent()
email_agent = EmailAgent()

supervisor = ReconFlowSupervisor(
    invoice_agent,
    po_agent,
    decision_agent,
    email_agent
)

result = supervisor.process_invoice("INV002")

In [26]:
# ======================================
# STEP 10 : AGENT CONVERSATION LOGGER
# ======================================

class AgentLogger:

    def __init__(self):
        self.logs = []

    def log(self, agent, message):

        self.logs.append({
            "time": datetime.now().strftime("%H:%M:%S"),
            "agent": agent,
            "message": message
        })

    def show(self):

        print("\n")
        print("="*70)
        print("RECONFLOW AI - AGENT REASONING TIMELINE")
        print("="*70)

        for log in self.logs:
            print(f"[{log['time']}] {log['agent']}: {log['message']}")

        print("="*70)

    def clear(self):
        self.logs = []

In [27]:
#TESTING THE AGENTS

invoice_agent = InvoiceAgent(invoices)
po_agent = POAgent(purchase_orders)
decision_agent = DecisionAgent()
email_agent = EmailAgent()

supervisor = ReconFlowSupervisor(
    invoice_agent,
    po_agent,
    decision_agent,
    email_agent
)

result = supervisor.process_invoice("INV002")



RECONFLOW AI - AGENT REASONING TIMELINE
[20:25:54] Supervisor: Started processing INV002
[20:25:54] Invoice Agent: Invoice extracted for vendor 'Global Tech'
[20:25:54] PO Agent: Matching Status: PARTIAL_MATCH
[20:25:54] PO Agent: Amount mismatch
[20:25:54] Decision Agent: Decision: ESCALATE
[20:25:54] Decision Agent: Confidence: 0.8
[20:25:54] Email Agent: Vendor email drafted successfully
[20:25:54] Supervisor: Workflow completed successfully


In [28]:
# =====================================
# STEP 11: CONFIGURATION LAYER
# =====================================

from dataclasses import dataclass

@dataclass
class Config:

    # Project Info
    PROJECT_NAME: str = "ReconFlow AI"
    VERSION: str = "2.0 (Agent Upgrade)"

    # LLM Settings
    LLM_PROVIDER: str = "mock"   # mock | ollama | gemini
    MODEL_NAME: str = "qwen2.5:7b"
    TEMPERATURE: float = 0.0
    MAX_TOKENS: int = 512

    # Business Rules
    CURRENCY: str = "INR"
    APPROVAL_THRESHOLD: float = 0.95
    ESCALATION_THRESHOLD: float = 0.80

    # Logging
    ENABLE_LOGS: bool = True

# Initialize config
config = Config()

In [29]:
# Testing the configuration layer

print("Project:", config.PROJECT_NAME)
print("Version:", config.VERSION)
print("LLM Provider:", config.LLM_PROVIDER)
print("Model:", config.MODEL_NAME)

Project: ReconFlow AI
Version: 2.0 (Agent Upgrade)
LLM Provider: mock
Model: qwen2.5:7b


In [30]:
# =====================================
# STEP 12: SHARED AGENT STATE
# =====================================

from dataclasses import dataclass, field
from typing import Dict, List, Any


@dataclass
class AgentState:

    # Core data
    invoice: Dict[str, Any] = field(default_factory=dict)
    po: Dict[str, Any] = field(default_factory=dict)

    # Processing results
    match_result: Dict[str, Any] = field(default_factory=dict)
    decision: Dict[str, Any] = field(default_factory=dict)
    email: Dict[str, Any] = field(default_factory=dict)

    # System tracking
    current_step: str = "INIT"
    logs: List[str] = field(default_factory=list)

    # Metadata
    invoice_id: str = ""
    vendor_risk_score: float = 0.0

In [31]:
#testing shared agent
state = AgentState()

state.invoice_id = "INV001"
state.current_step = "INVOICE_LOADED"

state.logs.append("Invoice loaded successfully")

print(state)

AgentState(invoice={}, po={}, match_result={}, decision={}, email={}, current_step='INVOICE_LOADED', logs=['Invoice loaded successfully'], invoice_id='INV001', vendor_risk_score=0.0)


In [33]:
# =====================================
# STEP 13: FINANCE TOOLS LAYER
# =====================================

class FinanceTools:

    def __init__(self, po_df):
        self.po_df = po_df

    # -------------------------
    # TOOL 1: PO LOOKUP
    # -------------------------
    def lookup_purchase_order(self, po_id):

        row = self.po_df[self.po_df["PO_ID"] == po_id]

        if row.empty:
            return {
                "found": False,
                "message": f"PO {po_id} not found"
            }

        row = row.iloc[0]

        return {
            "found": True,
            "po_id": row["PO_ID"],
            "vendor": row["Vendor"],
            "amount": float(row["Amount"]),
            "status": row["Status"]
        }

    # -------------------------
    # TOOL 2: VENDOR VALIDATION
    # -------------------------
    def validate_vendor(self, vendor_name):

        valid_vendors = self.po_df["Vendor"].unique().tolist()

        return {
            "is_valid": vendor_name in valid_vendors,
            "vendor": vendor_name,
            "known_vendors": valid_vendors
        }

    # -------------------------
    # TOOL 3: AMOUNT CHECK
    # -------------------------
    def check_amount(self, invoice_amount, po_amount, tolerance=0.01):

        diff = abs(invoice_amount - po_amount)

        return {
            "match": diff <= tolerance,
            "difference": diff,
            "invoice_amount": invoice_amount,
            "po_amount": po_amount
        }

In [34]:
tools = FinanceTools(purchase_orders)

# Test PO lookup
print(tools.lookup_purchase_order("PO1002"))

# Test vendor validation
print(tools.validate_vendor("Global Tech"))

# Test amount check
print(tools.check_amount(30000, 28000))

{'found': True, 'po_id': 'PO1002', 'vendor': 'Global Tech', 'amount': 28000.0, 'status': 'Approved'}
{'is_valid': True, 'vendor': 'Global Tech', 'known_vendors': ['ABC Electronics', 'Global Tech', 'Office World', 'Cloud Solutions', 'Dell India']}
{'match': False, 'difference': 2000, 'invoice_amount': 30000, 'po_amount': 28000}


In [36]:
# =====================================
# STEP 14: OLLAMA CLIENT
# =====================================

import requests
import json

class OllamaClient:

    def __init__(self, model="qwen2.5:7b"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):

        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False
        }

        response = requests.post(self.url, json=payload)

        if response.status_code != 200:
            return {
                "error": "Ollama not running or model not available"
            }

        return response.json()["response"]

In [40]:
# =====================================
# STEP 14: LLM DECISION AGENT
# =====================================

class LLMDecisionAgent:

    def __init__(self, llm_client):
        self.llm = llm_client

    def decide(self, state, tools_output):

        prompt = f"""
You are a Senior Finance AI Agent.

Your job is to decide whether to APPROVE, REJECT, or ESCALATE an invoice.

Return ONLY valid JSON.

---

INVOICE:
{state.invoice}

PURCHASE ORDER:
{state.po}

TOOL RESULTS:
{tools_output}

---

RULES:
- APPROVE → everything matches
- REJECT → major mismatch or invalid PO
- ESCALATE → minor mismatch or policy issue

Return format:
{{
  "decision": "",
  "confidence": 0-1,
  "reasoning": ""
}}
"""

        raw_output = self.llm.generate(prompt)

        try:
            return json.loads(raw_output)
        except:
            return {
                "decision": "ESCALATE",
                "confidence": 0.5,
                "reasoning": "LLM output parsing failed",
                "raw_output": raw_output
            }

In [42]:
#OLLAMA Fallback safe version

class MockLLM:

    def generate(self, prompt):
        return json.dumps({
            "decision": "ESCALATE",
            "confidence": 0.82,
            "reasoning": "Amount mismatch detected between invoice and PO"
        })

llm = MockLLM()

agent = LLMDecisionAgent(llm)

state = AgentState()
state.invoice = {"id": "INV002", "amount": 30000}
state.po = {"id": "PO1002", "amount": 28000}

tools_output = {
    "amount_check": {"match": False, "difference": 2000}
}

result = agent.decide(state, tools_output)

print(result)

{'decision': 'ESCALATE', 'confidence': 0.82, 'reasoning': 'Amount mismatch detected between invoice and PO'}


In [44]:
# =====================================
# STEP 15: TOOL CALLING AGENT LOOP
# =====================================

import json

class ToolCallingAgent:

    def __init__(self, tools, llm):
        self.tools = tools
        self.llm = llm

    def plan(self, state):

        prompt = f"""
You are an AI Finance Agent.

Decide which tools you need to solve this invoice.

TOOLS AVAILABLE:
1. lookup_purchase_order(po_id)
2. validate_vendor(vendor)
3. check_amount(invoice_amount, po_amount)

INVOICE:
{state.invoice}

PURCHASE ORDER (if known):
{getattr(state, "po", None)}

Return ONLY JSON:
{{
  "actions": [
    {{"tool": "", "input": {{}} }}
  ]
}}
"""

        response = self.llm.generate(prompt)

        # safer JSON parsing
        try:
            return json.loads(response)
        except Exception:
            return {
                "actions": [
                    {
                        "tool": "lookup_purchase_order",
                        "input": {
                            "po_id": state.invoice.get("po_id", None)
                        }
                    }
                ]
            }

    def execute(self, plan, state):

        results = {}

        # ensure logs exist
        if not hasattr(state, "logs"):
            state.logs = []

        for action in plan.get("actions", []):

            tool_name = action.get("tool")
            tool_input = action.get("input", {})

            if tool_name == "lookup_purchase_order":

                po_id = tool_input.get("po_id")
                if po_id is None:
                    continue

                result = self.tools.lookup_purchase_order(po_id)

                state.po = result
                results["po_lookup"] = result

            elif tool_name == "validate_vendor":

                vendor = tool_input.get("vendor")
                if vendor is None:
                    continue

                result = self.tools.validate_vendor(vendor)
                results["vendor_check"] = result

            elif tool_name == "check_amount":

                invoice_amount = tool_input.get("invoice_amount")
                po_amount = tool_input.get("po_amount")

                if invoice_amount is None or po_amount is None:
                    continue

                result = self.tools.check_amount(invoice_amount, po_amount)
                results["amount_check"] = result

            state.logs.append(f"Tool executed: {tool_name}")

        return results

In [45]:
#MOCK LLM
class MockPlannerLLM:

    def generate(self, prompt):

        return json.dumps({
            "actions": [
                {
                    "tool": "lookup_purchase_order",
                    "input": {"po_id": "PO1002"}
                },
                {
                    "tool": "check_amount",
                    "input": {
                        "invoice_amount": 30000,
                        "po_amount": 28000
                    }
                }
            ]
        })

In [46]:
#TEST TOOL

# Setup
tools = FinanceTools(purchase_orders)
llm = MockPlannerLLM()

agent = ToolCallingAgent(tools, llm)

state = AgentState()
state.invoice = {
    "invoice_id": "INV002",
    "po_id": "PO1002",
    "amount": 30000
}

# Run planning
plan = agent.plan(state)
print("PLAN:", plan)

# Execute tools
results = agent.execute(plan, state)
print("\nTOOL RESULTS:", results)

print("\nUPDATED STATE PO:", state.po)
print("\nLOGS:", state.logs)

PLAN: {'actions': [{'tool': 'lookup_purchase_order', 'input': {'po_id': 'PO1002'}}, {'tool': 'check_amount', 'input': {'invoice_amount': 30000, 'po_amount': 28000}}]}

TOOL RESULTS: {'po_lookup': {'found': True, 'po_id': 'PO1002', 'vendor': 'Global Tech', 'amount': 28000.0, 'status': 'Approved'}, 'amount_check': {'match': False, 'difference': 2000, 'invoice_amount': 30000, 'po_amount': 28000}}

UPDATED STATE PO: {'found': True, 'po_id': 'PO1002', 'vendor': 'Global Tech', 'amount': 28000.0, 'status': 'Approved'}

LOGS: ['Tool executed: lookup_purchase_order', 'Tool executed: check_amount']


In [48]:
# =====================================
# STEP 16: RAG - POLICY DATABASE
# =====================================

finance_policies = [
    {
        "id": "P1",
        "text": "Invoices with amount mismatch greater than 5% must be escalated for manual review."
    },
    {
        "id": "P2",
        "text": "All invoices must match an approved Purchase Order before payment."
    },
    {
        "id": "P3",
        "text": "Vendor mismatch cases must be rejected unless prior approval exists."
    },
    {
        "id": "P4",
        "text": "PO status must be 'Approved' before invoice processing."
    }
]

In [49]:
#Retriever

class PolicyRetriever:

    def __init__(self, policies):
        self.policies = policies

    def retrieve(self, query):

        query = query.lower()
        relevant = []

        for p in self.policies:

            if any(word in p["text"].lower() for word in query.split()):

                relevant.append(p)

        # fallback: return all if nothing matches
        if not relevant:
            relevant = self.policies

        return relevant

In [50]:
class PolicyRetriever:

    def __init__(self, policies):
        self.policies = policies

    def retrieve(self, query):

        query = query.lower()
        relevant = []

        for p in self.policies:

            if any(word in p["text"].lower() for word in query.split()):

                relevant.append(p)

        # fallback: return all if nothing matches
        if not relevant:
            relevant = self.policies

        return relevant

In [51]:
#test retrievr
retriever = PolicyRetriever(finance_policies)

retriever.retrieve("amount mismatch PO")

[{'id': 'P1',
  'text': 'Invoices with amount mismatch greater than 5% must be escalated for manual review.'},
 {'id': 'P3',
  'text': 'Vendor mismatch cases must be rejected unless prior approval exists.'},
 {'id': 'P4',
  'text': "PO status must be 'Approved' before invoice processing."}]

In [66]:
#RAG enhanced decision agent

class RAGDecisionAgent:

    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever

    def decide(self, state, tool_results):

        # Step 1: retrieve policies
        context_query = f"{state.invoice} {state.po}"
        policies = self.retriever.retrieve(context_query)

        policy_text = "\n".join([p["text"] for p in policies])

        # Step 2: build prompt
        prompt = f"""
You are a Finance Compliance AI Agent.

You MUST follow company policies strictly.

---

POLICIES:
{policy_text}

---

INVOICE:
{state.invoice}

PURCHASE ORDER:
{state.po}

TOOL RESULTS:
{tool_results}

---

Return ONLY JSON:
{{
  "decision": "APPROVE | REJECT | ESCALATE",
  "confidence": 0-1,
  "reasoning": "must include policy reference"
}}
"""

        response = self.llm.generate(prompt)

        try:
            return json.loads(response)
        except:

           return {
    "invoice_id": state.invoice.get("invoice_id", ""),
    "decision": decision["decision"] if isinstance(decision, dict) else "ESCALATE",
    "confidence": decision.get("confidence", 0.5) if isinstance(decision, dict) else 0.5,
    "reasoning": decision.get("reasoning", "No reasoning provided") if isinstance(decision, dict) else "Error"
}

In [53]:
#Test RAG Agent

class MockLLM:
    def generate(self, prompt):
        return json.dumps({
            "decision": "ESCALATE",
            "confidence": 0.88,
            "reasoning": "Amount mismatch exceeds 5% threshold as per policy P1"
        })

llm = MockLLM()
retriever = PolicyRetriever(finance_policies)

rag_agent = RAGDecisionAgent(llm, retriever)

state = AgentState()
state.invoice = {"id": "INV002", "amount": 30000}
state.po = {"id": "PO1002", "amount": 28000}

tool_results = {
    "amount_check": {"match": False, "difference": 2000}
}

result = rag_agent.decide(state, tool_results)

print(result)


{'decision': 'ESCALATE', 'confidence': 0.88, 'reasoning': 'Amount mismatch exceeds 5% threshold as per policy P1'}


In [56]:
# =====================================
# STEP 17: STREAMLIT DASHBOARD
# =====================================

import streamlit as st

st.set_page_config(page_title="ReconFlow AI", layout="wide")

st.title("🧠 ReconFlow AI - Finance Agent System")

st.write("Multi-Agent Invoice Reconciliation Platform")

# ----------------------------
# Load system (assumes notebook logic imported)
# ----------------------------

st.sidebar.header("Controls")

invoice_id = st.sidebar.text_input("Enter Invoice ID", "INV002")

run_button = st.sidebar.button("Run AI Agents")

# ----------------------------
# Run Pipeline
# ----------------------------

if run_button:

    st.subheader("🔄 Processing Invoice")

    # Step 1: Invoice Agent
    invoice = invoice_agent.extract_invoice(invoice_id)
    st.json(invoice)

    # Step 2: PO Agent
    match = po_agent.match_invoice_with_po(invoice)
    st.subheader("📑 PO Matching Result")
    st.json(match)

    # Step 3: Decision Agent
    decision = decision_agent.decide(match)
    st.subheader("🧠 AI Decision")
    st.json(decision)

    # Step 4: Email Agent
    email = email_agent.generate_email(decision)
    st.subheader("📧 Generated Email")

    st.write("**Subject:**", email["subject"])
    st.text(email["body"])

    # ----------------------------
    # Reasoning Timeline (if logger exists)
    # ----------------------------
    st.subheader("📊 Agent Reasoning Timeline")

    try:
        supervisor_result = supervisor.process_invoice(invoice_id)
        st.json(supervisor_result["logs"])
    except:
        st.warning("Run supervisor version for full logs")

2026-06-20 21:43:18.204 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 21:43:18.206 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 21:43:18.663 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-20 21:43:18.663 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 21:43:18.665 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 21:43:18.666 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 21:43:18.667 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [58]:
# =====================================
# STEP 18: ANALYTICS ENGINE
# =====================================

class AnalyticsEngine:

    def __init__(self):
        self.records = []

    def log_result(self, invoice_id, decision, match_result):

        self.records.append({
            "invoice_id": invoice_id,
            "decision": decision["decision"],
            "confidence": decision.get("confidence", 0),
            "status": match_result.get("status", "UNKNOWN"),
            "issues": match_result.get("issues", [])
        })

    def summary(self):

        total = len(self.records)

        if total == 0:
            return {"message": "No data available"}

        approved = sum(1 for r in self.records if r["decision"] == "APPROVE")
        rejected = sum(1 for r in self.records if r["decision"] == "REJECT")
        escalated = sum(1 for r in self.records if r["decision"] == "ESCALATE")

        mismatch = sum(1 for r in self.records if r["status"] != "MATCH")

        return {
            "total_invoices": total,
            "approved": approved,
            "rejected": rejected,
            "escalated": escalated,
            "approval_rate": round(approved / total, 2),
            "escalation_rate": round(escalated / total, 2),
            "mismatch_rate": round(mismatch / total, 2)
        }

    def show(self):

        import pprint
        pprint.pprint(self.summary())

In [73]:
# =====================================
# FINAL PIPELINE RUNNER
# =====================================

def run_pipeline(invoice_id):

    print("\n🚀 STARTING RECONFLOW AI PIPELINE\n")

    # ----------------------------
    # Step 1: Invoice
    # ----------------------------
    invoice = invoice_agent.extract_invoice(invoice_id)
    print("📄 Invoice Loaded:", invoice)

    # ----------------------------
    # Step 2: PO Matching
    # ----------------------------
    match = po_agent.match_invoice_with_po(invoice)
    print("\n📑 PO Match:", match)

    # ----------------------------
    # Step 3: Tool Execution (if using Step 15 agent)
    # ----------------------------
    state = AgentState()
    state.invoice = invoice

    tools = FinanceTools(purchase_orders)
    llm = MockPlannerLLM()
    tool_agent = ToolCallingAgent(tools, llm)

    plan = tool_agent.plan(state)
    tool_results = tool_agent.execute(plan, state)

    print("\n🧰 Tool Results:", tool_results)

    # ----------------------------
    # Step 4: RAG Decision
    # ----------------------------
    retriever = PolicyRetriever(finance_policies)
    rag_llm = MockLLM()
    decision_agent = RAGDecisionAgent(rag_llm, retriever)

    decision = decision_agent.decide(state, tool_results)
    print("\n🧠 Decision:", decision)

    state.decision = decision

    # ----------------------------
    # Step 5: Email
    # ----------------------------
    decision["invoice_id"] = invoice["invoice_id"]
    email = email_agent.generate_email(decision)
    print("\n📧 Email Generated:")
    print(email)

    # ----------------------------
    # Step 6: Analytics
    # ----------------------------
    analytics.log_result(invoice_id, decision, match)

    print("\n📊 UPDATED ANALYTICS:")
    analytics.show()

    print("\n✅ PIPELINE COMPLETED SUCCESSFULLY\n")

    return {
        "invoice": invoice,
        "match": match,
        "tool_results": tool_results,
        "decision": decision,
        "email": email
    }

In [1]:
result = run_pipeline("INV002")

NameError: name 'run_pipeline' is not defined